# Lab 1 · Catch the fraud — Supervised Learning

**Fundamentals of AI · ~25 min** · Dataset: [Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
— 284,807 real card transactions, only 492 of them fraud.

## 1 · Setup & load (2 min)

Three quick cells: **theme** (dark, matches the slides) → **load data** (auto-download, no login) → **peek**.

In [ ]:
#@title 1a. Theme + plot_cm() helper { display-mode: "form" }
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix

BG, ACCENT, ACCENT2, TEXT = "#030503", "#00E676", "#3DFF8A", "#FFFFFF"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "axes.titlecolor": TEXT,
    "grid.color": "#1c3a24", "font.size": 14, "axes.titlesize": 18,
    "axes.labelsize": 15, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.facecolor": "#0a140d", "legend.edgecolor": ACCENT,
    "figure.figsize": (8, 5),
})

def plot_cm(y_true, y_pred, title=""):
    """Dark-themed confusion matrix. Rows = truth, cols = prediction."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots()
    ax.imshow(cm, cmap="Greens")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["legit", "fraud"], color=TEXT)
    ax.set_yticklabels(["legit", "fraud"], color=TEXT)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actually")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                    color=TEXT, fontsize=18, fontweight="bold")
    plt.tight_layout(); plt.show()

def need(*names):
    """Friendly stop if an earlier step hasn't run yet."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise SystemExit(f"✏️ Not ready yet — missing: {', '.join(missing)}. Run the earlier cells first (or open 'Show solution').")
print("Theme ready.")

In [ ]:
#@title 1b. Load the data { display-mode: "form" }
import os
try:
    from sklearn.datasets import fetch_openml
    data = fetch_openml(data_id=1597, as_frame=True, parser="auto")   # OpenML mirror, no login
    X_all = data.data if hasattr(data, "data") else data[0]
    y_all = (data.target if hasattr(data, "target") else data[1]).astype(int).values
except Exception as e:
    print("OpenML download failed:", e)
    if os.path.exists("creditcard.csv"):
        print("Found creditcard.csv — loading it instead.")
        _df = pd.read_csv("creditcard.csv")
        X_all = _df.drop(columns=["Class"]); y_all = _df["Class"].astype(int).values
    else:
        print("FALLBACK: download creditcard.csv from")
        print("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  (free login),")
        print("upload it via the folder icon on the left, then re-run this cell.")
        raise SystemExit("Dataset not loaded yet.")
print("Loaded OK.")

In [ ]:
#@title 1c. Peek at the data { display-mode: "form" }
print(f"Shape: {X_all.shape[0]:,} rows × {X_all.shape[1]} columns")
print("Columns:", ", ".join(X_all.columns[:5]), "...,", ", ".join(X_all.columns[-3:]))
X_all.head()

> **If 1b fails:** download `creditcard.csv` from the [Kaggle page](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
> (free login), upload it in Colab's file panel, re-run 1b. Columns `V1`–`V28` are anonymized
> PCA features — the model doesn't need to know what they mean.

## 2 · How rare is fraud? (3 min)

> 📝 **Predict-then-run:** what % of transactions are fraud? Guess first, *then* run the next two cells.

In [ ]:
#@title 2a. The numbers { display-mode: "form" }
counts = pd.Series(y_all).value_counts().sort_index()
rate = counts.get(1, 0) / len(y_all) * 100
print(f"Fraud: {counts.get(1,0):,}  ({rate:.3f}%)")
print(f"Legit: {counts.get(0,0):,}  ({100-rate:.3f}%)")
print(f"Ratio: ~1 fraud per {counts.get(0,0)/max(counts.get(1,0),1):,.0f} legit transactions")

In [ ]:
#@title 2b. Class balance chart { display-mode: "form" }
fig, ax = plt.subplots()
bars = ax.bar(["Legit (0)", "Fraud (1)"], [counts.get(0,0), counts.get(1,0)],
              color=[ACCENT2, "#FF5252"], width=0.5)
ax.set_yscale("log")   # log scale — otherwise the fraud bar is invisible
ax.set_ylabel("Transactions (log scale)")
ax.set_title("Class balance — fraud is a needle in a haystack")
for b, v in zip(bars, [counts.get(0,0), counts.get(1,0)]):
    ax.text(b.get_x()+b.get_width()/2, v*1.2, f"{v:,}", ha="center", fontsize=14)
plt.tight_layout(); plt.show()

<details><summary>💡 Why does this matter?</summary>

A model that always says "legit" is right ~99.8% of the time — and catches **zero** fraud.
Accuracy alone can't spot a lazy model. Next section: the baseline trap.</details>

## 3 · The baseline trap (3 min)

The laziest model possible: **always predict "legit"**.

> 📝 **Predict-then-run:** what accuracy will it get? What about frauds caught?

In [ ]:
#@title 3a. Always-legit model — scores { display-mode: "form" }
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")   # always guess the majority class
dummy.fit(X_all, y_all)
pred_dummy = dummy.predict(X_all)

acc = accuracy_score(y_all, pred_dummy)
print(f"Accuracy: {acc*100:.2f}%   ← looks amazing!")
print(f"Frauds caught: {(pred_dummy==1).sum()} / {int(y_all.sum())}")

In [ ]:
#@title 3b. Its confusion matrix { display-mode: "form" }
plot_cm(y_all, pred_dummy, title=f"Always-legit — accuracy {acc*100:.1f}%, frauds caught: 0")

**Why accuracy lies:** 99.8% of transactions are legit, so "always legit" scores ~99.8%
while catching 0% of fraud. On imbalanced data we need **precision** (of our alarms, how many
are real?) and **recall** (of real frauds, how many caught?).

<details><summary>💡 Checkpoint — which do we want for fraud?</summary>

A **missed fraud** costs ~$500; a **false alarm** costs an analyst ~5 min (~$5).
So we want high **recall** — catch nearly every fraud — even at the price of more false alarms.</details>

## 4 · Train/test split + first model (5 min)

Split the data: the model **trains** on one part, then takes an **exam** on data it never saw.

### ✏️ Blank #1 — make the split *stratified*

`train_test_split` picks random rows — but we want the fraud ratio (~0.17%) identical in both sets.
Fill in the placeholder (hint: there's a parameter for exactly this).

In [ ]:
#@title 4. Train/test split — BLANK #1 { display-mode: "form" }
from sklearn.model_selection import train_test_split

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42,
        stratify=YOUR_ANSWER_HERE,     # ← replace YOUR_ANSWER_HERE (the target variable)
    )
except Exception as e:
    print("✏️ Blank #1 not filled correctly — see 'Show solution' below, then re-run.")
    print(f"   ({e})")

if "X_train" in globals():
    print(f"Train: {X_train.shape[0]:,}  ({y_train.mean()*100:.3f}% fraud)")
    print(f"Test:  {X_test.shape[0]:,}  ({y_test.mean()*100:.3f}% fraud)")

In [ ]:
#@title Show solution { display-mode: "form" }
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42,
    stratify=y_all,          # keeps the same fraud ratio in both sets
)
print("Split done — stratified.")

## 5 · Logistic Regression — train, score, plot (4 min)

A classic fast model that outputs a fraud *probability* per transaction. `StandardScaler`
rescaled `Amount`/`Time` so they don't dominate the PCA features.

In [ ]:
#@title 5a. Train logistic regression { display-mode: "form" }
need("X_train")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print("Trained. Moving on to scores...")

In [ ]:
#@title 5b. Precision & recall { display-mode: "form" }
need("pred_lr")
from sklearn.metrics import precision_score, recall_score, classification_report

prec = precision_score(y_test, pred_lr, zero_division=0)
rec  = recall_score(y_test, pred_lr)
print(f"Logistic Regression — precision: {prec:.3f}   recall: {rec:.3f}")
print()
print(classification_report(y_test, pred_lr, target_names=["legit","fraud"], digits=3))

In [ ]:
#@title 5c. Confusion matrix { display-mode: "form" }
need("pred_lr", "rec")
plot_cm(y_test, pred_lr, title=f"Logistic Regression — recall {rec:.2f}")

<details><summary>💡 Checkpoint — read the matrix</summary>

Top-right = **false alarms** (legit flagged as fraud). Bottom-left = **missed frauds**.
Which is worse here? (Missed fraud ~$500 vs false alarm ~$5 → bottom-left hurts most → we want recall up.)</details>

## 6 · Trees, forests & class weights (4 min)

A **decision tree** asks yes/no questions about features until it reaches a verdict.
A **random forest** averages hundreds of trees — usually stronger.

In [ ]:
#@title 6a. Decision tree — overfitting knob { display-mode: "form" }
from sklearn.tree import DecisionTreeClassifier

max_depth = 10          # 🌳 KNOB — try 2, 5, 10, None and re-run

tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
tree.fit(X_train, y_train)

print(f"max_depth = {max_depth}")
print(f"Train accuracy: {tree.score(X_train, y_train)*100:.1f}%")
print(f"Test  accuracy: {tree.score(X_test,  y_test )*100:.1f}%")
print("(train ≫ test → the tree memorized the data = overfitting)")

Fraud is only 0.17% of the data, so the model can score well while ignoring fraud.
**`class_weight="balanced"`** tells it: *fraud cases count more during training.*

### ✏️ Blank #2 — add one line to the forest below

In [ ]:
#@title 6b. Random forest — BLANK #2 { display-mode: "form" }
need("X_train")
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    # class_weight="balanced",   # ← YOUR CODE HERE: uncomment/add this line
    n_jobs=-1,
)
rf.fit(X_train, y_train)
if getattr(rf, "class_weight", None) is None:
    print("ℹ️  Blank #2 empty — forest trained WITHOUT class weights.")
else:
    print("Forest trained WITH class_weight='balanced'.")

In [ ]:
#@title Show solution { display-mode: "form" }
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=50, max_depth=12, random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print("Done — class_weight='balanced' active.")

In [ ]:
#@title 6c. Forest scores { display-mode: "form" }
need("rf")
pred_rf = rf.predict(X_test)
prec_rf = precision_score(y_test, pred_rf, zero_division=0)
rec_rf  = recall_score(y_test, pred_rf)
print(f"Random Forest — precision: {prec_rf:.3f}   recall: {rec_rf:.3f}")

In [ ]:
#@title 6d. Forest confusion matrix { display-mode: "form" }
need("pred_rf")
plot_cm(y_test, pred_rf, title="Random Forest (class_weight='balanced')")

In [ ]:
#@title 6e. All models compared { display-mode: "form" }
need("pred_lr", "pred_rf", "dummy", "X_test")
from sklearn.metrics import accuracy_score

models = {"Always-legit": dummy.predict(X_test),
          "Logistic Reg.": pred_lr, "Random Forest": pred_rf}
rows = [{"Model": k,
         "Accuracy": accuracy_score(y_test, p),
         "Precision": precision_score(y_test, p, zero_division=0),
         "Recall": recall_score(y_test, p)}
        for k, p in models.items()]
table = pd.DataFrame(rows).set_index("Model")
display(table.style.format({"Accuracy":"{:.3f}","Precision":"{:.3f}","Recall":"{:.3f}"})
              .highlight_max(subset=["Recall"], color=ACCENT2))

<details><summary>💡 What to look for</summary>

Baseline has the best *accuracy* — worst *recall* (0%). Forest+weights catches far more
fraud than plain logistic regression, at the cost of more false alarms. That trade-off is
what we tune next.</details>

## 7 · The cost game (4 min)

The model outputs a fraud **probability**; we pick a **threshold** — above it → "fraud".
Missed fraud = **$500**, false alarm = **$5**. Lower threshold → more frauds caught, more false alarms.

In [ ]:
#@title 7a. Probabilities + best threshold { display-mode: "form" }
need("rf")
COST_FRAUD_MISSED, COST_FALSE_ALARM = 500, 5
proba = rf.predict_proba(X_test)[:, 1]
yv = np.asarray(y_test)

grid = np.linspace(0.01, 0.99, 197)
costs = [(((yv==1)&((proba>=t).astype(int)==0)).sum()*COST_FRAUD_MISSED +
          ((yv==0)&((proba>=t).astype(int)==1)).sum()*COST_FALSE_ALARM) for t in grid]
best_t = grid[int(np.argmin(costs))]
print(f"Sweeping all thresholds: cheapest is {best_t:.2f} → ${min(costs):,}")
print("Now drag the slider — does 0.5 win?")

In [ ]:
#@title 7b. Threshold slider { display-mode: "form" }
need("proba")
import ipywidgets as widgets

def total_cost(threshold):
    p = (proba >= threshold).astype(int)
    missed = int(((yv == 1) & (p == 0)).sum())
    false  = int(((yv == 0) & (p == 1)).sum())
    return (f"threshold = {threshold:.2f}\n"
            f"  missed frauds: {missed}  →  ${missed*COST_FRAUD_MISSED:,}\n"
            f"  false alarms:  {false}  →  ${false*COST_FALSE_ALARM:,}\n"
            f"  TOTAL COST:    ${missed*COST_FRAUD_MISSED + false*COST_FALSE_ALARM:,}")

out = widgets.Output()
slider = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01,
                             description="threshold", style={"description_width":"initial"})
slider.observe(lambda ch: (out.clear_output(wait=True),
                           out.append_stdout(total_cost(ch["new"]))), names="value")
with out:
    print(total_cost(0.5))
display(widgets.VBox([slider, out]))

<details><summary>💡 Why isn't the best threshold 0.5?</summary>

0.5 ignores **costs**. A missed fraud costs 100× a false alarm, so the cheapest threshold
is usually *lower* (~0.2): happily investigate many legit transactions to catch one more fraud.</details>

## 8 · Takeaways (1 min)

- Fraud is rare (0.17%) → **accuracy lies**; use precision & recall.
- **Stratified split** keeps class balance; **class weights** make the model care about fraud.
- Pick the **threshold by cost**, not habit. Trees overfit — watch train vs test score.

🎉 Lab 1 complete.

<details><summary>🔑 Solutions</summary>

- **Blank #1:** `stratify=y_all`
- **Blank #2:** `class_weight="balanced",`
</details>